In [1]:
import pandas as pd
import numpy as np

# 🌟 设置随机种子，确保你我的靶场物理对齐
np.random.seed(42)

# 1. 模拟上游分发系统：生成绝对唯一、且带有物理落盘乱序的 log_id
log_ids = [f"log_20260621_{i:04d}" for i in range(1, 11)]

data = {
    'log_id': log_ids,  # 👈 刚性补上你指出的上游数据排序/唯一标签
    'user_id': [101, 102, 101, 103, 101, 102, 103, 101, 102, 101],
    'click_time': [
        '2026-06-21 10:00:00',  # 101-1
        '2026-06-21 10:05:00',  # 102-1
        '2026-06-21 10:15:00',  # 101-2 (间隔 15m)
        '2026-06-21 10:20:00',  # 103-1 
        '2026-06-21 11:00:00',  # 101-3 (🚨 距离上一次 45m，流失卡点！)
        '2026-06-21 10:05:00',  # 102-2 (🚨 极端边界：与102-1同一秒点击！)
        '2026-06-21 11:30:00',  # 103-2 (🚨 距离上一次 70m，流失卡点！)
        '2026-06-21 11:00:00',  # 101-4 (🚨 极端边界：与101-3同一秒点击！)
        '2026-06-21 12:00:00',  # 102-3 (🚨 距离上一次 115m，流失卡点！)
        '2026-06-21 13:00:00'   # 101-5 (🚨 距离上一次 120m，流失卡点！)
    ],
    'page_id': ['home', 'home', 'search', 'home', 'detail', 'cart', 'detail', 'home', 'payment', 'logout']
}

click_logs = pd.DataFrame(data)

# 2. 模拟分布式洗牌（Shuffle）：将物理行顺序彻底打乱，粉碎一切时间连续性
click_logs = click_logs.sample(frac=1, random_state=2026).reset_index(drop=True)

print("--- ⚔️ 升级版工业级原始混乱数据表 click_logs ---")
print(click_logs)

--- ⚔️ 升级版工业级原始混乱数据表 click_logs ---
              log_id  user_id           click_time  page_id
0  log_20260621_0008      101  2026-06-21 11:00:00     home
1  log_20260621_0004      103  2026-06-21 10:20:00     home
2  log_20260621_0010      101  2026-06-21 13:00:00   logout
3  log_20260621_0009      102  2026-06-21 12:00:00  payment
4  log_20260621_0005      101  2026-06-21 11:00:00   detail
5  log_20260621_0006      102  2026-06-21 10:05:00     cart
6  log_20260621_0001      101  2026-06-21 10:00:00     home
7  log_20260621_0003      101  2026-06-21 10:15:00   search
8  log_20260621_0007      103  2026-06-21 11:30:00   detail
9  log_20260621_0002      102  2026-06-21 10:05:00     home


### 🛠️ 核心任务：

找出每个用户连续两次点击之间，时间间隔严格大于 30 分钟（1800秒）的记录。

### 📊 期待交付的结果矩阵字段：

你的 SQL 和 Pandas 最终输出的表格中，必须严格包含以下列：

1. `user_id`：用户ID
    
2. `current_log_id`：**当前触发流失的那次点击的 `log_id`**（用来进行数据追溯）
    
3. `click_time`：当前点击的物理时间
    
4. `next_click_time`：下一次点击的物理时间
    
5. `diff_seconds`：计算出来的精准秒数差值

In [ ]:
# SQL轨道
import sqlite3
conn = sqlite3.connect(':memory:')
click_logs.to_sql('click_logs',conn,if_exists='replace',index=False)
sql_query = """
WITH next_clicks AS (
    SELECT  user_id,
            log_id AS current_log_id,
            click_time,
            LEAD(click_time, 1) OVER(
                PARTITION BY user_id 
                ORDER BY click_time, log_id
            ) AS next_click_time
    FROM click_logs
),
time_calculation AS (
    SELECT  user_id,
            current_log_id,
            click_time,
            next_click_time,
            (CAST(strftime('%s', datetime(next_click_time)) AS INTEGER) - 
             CAST(strftime('%s', datetime(click_time)) AS INTEGER)) AS diff_seconds
    FROM next_clicks
    WHERE next_click_time IS NOT NULL
)
SELECT  user_id,
        current_log_id,
        click_time,
        next_click_time,
        diff_seconds
FROM time_calculation
WHERE diff_seconds > 1800
  AND click_time >= '2026-06-21 00:00:00'  -- 🛡️ 在这里过滤！确保错位计算完后再切片
  AND click_time < '2026-06-22 00:00:00'
ORDER BY user_id, click_time;
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

   user_id     current_log_id           click_time      next_click_time  \
0      101  log_20260621_0003  2026-06-21 10:15:00  2026-06-21 11:00:00   
1      101  log_20260621_0008  2026-06-21 11:00:00  2026-06-21 13:00:00   
2      102  log_20260621_0006  2026-06-21 10:05:00  2026-06-21 12:00:00   
3      103  log_20260621_0004  2026-06-21 10:20:00  2026-06-21 11:30:00   

   diff_seconds  
0          2700  
1          7200  
2          6900  
3          4200  


In [22]:
# PANDAS轨道
df_click_logs = click_logs.copy()
df_click_logs['click_time'] = pd.to_datetime(df_click_logs['click_time'])
df_click_logs = df_click_logs.sort_values(by=['click_time','log_id']).reset_index(drop=True)

df_click_logs['next_click_time'] = df_click_logs.groupby('user_id')['click_time'].shift(-1)
df_click_logs['diff_seconds'] = (df_click_logs['next_click_time'] - df_click_logs['click_time']).dt.total_seconds()

df_click_logs = df_click_logs.rename(columns={'log_id':'current_log_id'})

final_df = df_click_logs[df_click_logs['diff_seconds'] > 1800][
    ['user_id', 'current_log_id', 'click_time', 'next_click_time', 'diff_seconds']
].reset_index(drop=True)

print(final_df)

   user_id     current_log_id          click_time     next_click_time  \
0      102  log_20260621_0006 2026-06-21 10:05:00 2026-06-21 12:00:00   
1      101  log_20260621_0003 2026-06-21 10:15:00 2026-06-21 11:00:00   
2      103  log_20260621_0004 2026-06-21 10:20:00 2026-06-21 11:30:00   
3      101  log_20260621_0008 2026-06-21 11:00:00 2026-06-21 13:00:00   

   diff_seconds  
0        6900.0  
1        2700.0  
2        4200.0  
3        7200.0  


In [ ]:
import os
import sqlite3
import pandas as pd

# =====================================================================
# 🛠️ 1. 刚性路径与物理文件初始化（完美保留你的设计）
# =====================================================================
current_dir = os.getcwd()
while os.path.basename(current_dir) != 'data-science-labs':
    parent_dir = os.path.dirname(current_dir)
    if parent_dir == current_dir:
        break
    current_dir = parent_dir
folder_path = os.path.join(current_dir, 'raw_data', 'datacamp', 'SQL', 'click_logs')
file_path = os.path.join(folder_path, 'click_logs.csv')

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
if os.path.exists(file_path):
    os.remove(file_path)

# =====================================================================
# 🛠️ 2. 纯内存沙盒环境初始化
# =====================================================================
conn = sqlite3.connect(':memory:')
click_logs.to_sql('click_logs', conn, if_exists='replace', index=False)

# 🌟 优化后的 SQL 大闸：把过滤挪到最外层，彻底解决跨天/跨周期截断漏洞
sql_query = """
WITH next_clicks AS (
    SELECT  user_id,
            log_id AS current_log_id,
            click_time,
            LEAD(click_time, 1) OVER(
                PARTITION BY user_id 
                ORDER BY click_time, log_id
            ) AS next_click_time
    FROM click_logs
),
time_calculation AS (
    SELECT  user_id,
            current_log_id,
            click_time,
            next_click_time,
            (CAST(strftime('%s', datetime(next_click_time)) AS INTEGER) - 
             CAST(strftime('%s', datetime(click_time)) AS INTEGER)) AS diff_seconds
    FROM next_clicks
    WHERE next_click_time IS NOT NULL
)
SELECT  user_id,
        current_log_id,
        click_time,
        next_click_time,
        diff_seconds
FROM time_calculation
WHERE diff_seconds > 1800
  AND click_time >= '2026-06-21 00:00:00'  -- 🛡️ 在这里过滤！确保错位计算完后再切片
  AND click_time < '2026-06-22 00:00:00'
ORDER BY user_id, click_time;
"""

# =====================================================================
# 🛠️ 3. 启用高压流式 ETL 落地管道（带闭环安全锁）
# =====================================================================
CHUNK_SIZE = 100000
total_records = 0

try:
    # 建立流式块读取生成器
    query_generator = pd.read_sql_query(sql_query, conn, chunksize=CHUNK_SIZE)

    for i, feature_chunk in enumerate(query_generator):
        if i == 0:
            # 第一批：创建新文件，写入表头
            feature_chunk.to_csv(file_path, mode='w', index=False)
        else:
            # 后续批次：纯追加模式，物理静音表头
            feature_chunk.to_csv(file_path, mode='a', header=False, index=False)
        
        current_rows = len(feature_chunk)
        total_records += current_rows
        print(
            f"⚡ [流式管道] 目前成功处理第 {i+1} 批分布式块数据...\n"
            f"   └─ 本批次放行 {current_rows} 条高危流失样本，累计落盘 {total_records} 条。"
        )

except Exception as e:
    print(f"🚨【重大安全事故】生产落盘管道遭遇不可抗力崩溃: {e}")
finally:
    # 🛡️ 无论如何，必须在物理内存中注销连接销毁沙盒，不留任何句柄泄露
    conn.close()

# =====================================================================
# 🛠️ 4. 终端最终验收对账
# =====================================================================
print("\n" + "="*50 + "\n🔥 正在从物理磁盘回显最终 ETL 审计文件...\n" + "="*50)
if os.path.exists(file_path):
    df_result = pd.read_csv(file_path)
    print(df_result.to_string(index=False))
else:
    print("🚨 致命错误：物理磁盘未检测到目标 click_logs.csv 资产！")

目前处理了1批数据该批次放行了4条数据，累计放行4条
 user_id    current_log_id          click_time     next_click_time  diff_seconds
     101 log_20260621_0003 2026-06-21 10:15:00 2026-06-21 11:00:00          2700
     101 log_20260621_0008 2026-06-21 11:00:00 2026-06-21 13:00:00          7200
     102 log_20260621_0006 2026-06-21 10:05:00 2026-06-21 12:00:00          6900
     103 log_20260621_0004 2026-06-21 10:20:00 2026-06-21 11:30:00          4200
